# Sorting and Navigating MultiIndexed DataFrames

When you set multiple columns as a MultiIndex, the rows will initially appear in whatever order they existed in the source DataFrame. This can result in disordered groupings (e.g., `'Europe'` appearing, followed by `'Asia'`, and then `'Europe'` again).

To ensure optimal performance and cleanly grouped layouts, you must **always sort your MultiIndex**. Sorting aligns your outer and inner indices alphabetically or numerically, making the table highly legible and preparing it for seamless lookup and slicing operations.

### Simple Explanation & Real-World Analogy
Imagine the filing cabinet again. If the drawers are labeled `'Europe'`, `'Asia'`, and `'Africa'` in a random, disorganized sequence, it takes longer to locate what you need. Sorting alphabetical drawer levels (`'Africa'`, then `'Asia'`, then `'Europe'`) and then sorting the country folders inside them alphabetically is exactly what `.sort_index()` does.

### Code Examples
Let's take our unsorted MultiIndex DataFrame from Module 31 and sort it.



In [4]:
import pandas as pd

# Create a sample DataFrame of global populations (mock data in millions)
data = {
    'Continent': ['Europe', 'Europe', 'Asia', 'Asia', 'Africa', 'Africa'],
    'Country': ['Albania', 'Germany', 'China', 'India', 'Angola', 'Egypt'],
    'Population': [2.8, 83.2, 1412.0, 1393.0, 32.8, 102.3],
    'Area_KM2': [28748, 357022, 9596961, 3287263, 1246700, 1001450]
}

df = pd.DataFrame(data)

# Turn the two text columns into the row labels (the MultiIndex).
# Without this line the index is just the row numbers 0..5, and it has only ONE level.
df.set_index(['Continent', 'Country'], inplace=True)

print("--- Unsorted MultiIndexed DataFrame ---")
print(df)

# Sort the MultiIndex alphabetically: outer level first, then the inner level
df.sort_index(inplace=True)
print("--- Sorted MultiIndexed DataFrame ---")
print(df)


--- Unsorted MultiIndexed DataFrame ---
                   Population  Area_KM2
Continent Country                      
Europe    Albania         2.8     28748
          Germany        83.2    357022
Asia      China        1412.0   9596961
          India        1393.0   3287263
Africa    Angola         32.8   1246700
          Egypt         102.3   1001450
--- Sorted MultiIndexed DataFrame ---
                   Population  Area_KM2
Continent Country                      
Africa    Angola         32.8   1246700
          Egypt         102.3   1001450
Asia      China        1412.0   9596961
          India        1393.0   3287263
Europe    Albania         2.8     28748
          Germany        83.2    357022


Notice how `Continent` has been sorted alphabetically (`Africa` ➡️ `Asia` ➡️ `Europe`), and within each continent, the countries are also sorted alphabetically (e.g., `Angola` ➡️ `Egypt`).

#### Sorting Specific Levels
By default, `.sort_index()` sorts all levels from outer to inner. However, you can choose to sort only a specific level or sort them differently (e.g., sorting one level ascending and another descending):

In [5]:
# Sort Continent ascendingly, but Country descendingly.
# `level` names the levels we sort by, and `ascending` gives one True/False per level.
# This only works because the index now really has 2 levels.
df.sort_index(level=['Continent', 'Country'], ascending=[True, False], inplace=True)
print(df)


                   Population  Area_KM2
Continent Country                      
Africa    Egypt         102.3   1001450
          Angola         32.8   1246700
Asia      India        1393.0   3287263
          China        1412.0   9596961
Europe    Germany        83.2    357022
          Albania         2.8     28748



###  Common Beginner Mistakes
1.  **Slicing Unsorted Indices**: If you try to select or slice a range of rows in a MultiIndex DataFrame that has not been sorted, Pandas will throw a loud `UnsortedIndexError`. Always chain or run `.sort_index()` immediately after calling `.set_index()` to prevent this.

#### Exercise 1 (Medium)
Given the sorted branch DataFrame from Module 31's exercise:
1. Reset the index using `.reset_index()`.
2. Create a new MultiIndex using `['Department', 'City']`.
3. Sort the DataFrame so that `Department` is sorted ascending (alphabetically) and `City` is sorted descending (alphabetically).


In [7]:
# 1. Reset index
branch_data = {
    'City': ['New York', 'New York', 'London', 'London'],
    'Department': ['Sales', 'Engineering', 'Sales', 'Engineering'],
    'Employees': [12, 45, 8, 38],
    'Budget': [150000, 450000, 110000, 400000]
}
df_branch = pd.DataFrame(branch_data)
df_flat = df_branch.reset_index()

# 2. Set new MultiIndex with Department as outer level
df_flat.set_index(['Department', 'City'], inplace=True)

# 3. Sort Department ascending, City descending
df_flat.sort_index(level=['Department', 'City'], ascending=[True, False], inplace=True)
print(df_flat)

                      index  Employees  Budget
Department  City                              
Engineering New York      1         45  450000
            London        3         38  400000
Sales       New York      0         12  150000
            London        2          8  110000


# Selecting and Slicing MultiIndexed Data

Once your DataFrame has a MultiIndex, standard lookup patterns change. To select individual data or slices, we use **`.loc`** (label-based location).

Selecting data from a MultiIndex requires defining labels for different levels of the index. You will learn how to:
*   Retrieve an entire group from the outer level.
*   Retrieve a specific nested record using a **tuple**.
*   Contrast why **`.iloc`** (integer location) ignores MultiIndex labels entirely and works on pure positional coordinates.

### 2. Simple Explanation & Real-World Analogy
*   **`.loc['Asia']`**: Opening the `'Asia'` drawer and taking everything out of it.
*   **`.loc[('Asia', 'China')]`**: Opening the `'Asia'` drawer, pulling out exactly the `'China'` folder, and reading its values.
*   **`.iloc[2]`**: Ignoring the drawer labels and folders entirely. You simply count down to the 3rd physical row of papers in the cabinet and pull it out.

### Code Examples
Let's work with our sorted global population DataFrame.


In [8]:
# Reset to default alphabetical sorted state
df.sort_index(inplace=True)

#### Selecting Entire Outer Groups with `.loc`
To grab all rows belonging to an outer index label, simply pass that label to `.loc`:

In [9]:
# Select all records belonging to Asia
asia_data = df.loc['Asia']
print("--- df.loc['Asia'] ---")
print(asia_data)

--- df.loc['Asia'] ---
         Population  Area_KM2
Country                      
China        1412.0   9596961
India        1393.0   3287263


*Notice: The outer index level (`Continent`) is dropped from the output, leaving `Country` as a clean single index.*

#### Selecting Specific Inner Rows using Tuples
To target a specific nested row, pass the hierarchical keys as a **tuple** `(outer_label, inner_label)` inside `.loc`:


In [10]:
# Select China specifically (nested under Asia)
china_stats = df.loc[('Asia', 'China')]
print("--- df.loc[('Asia', 'China')] ---")
print(china_stats)

--- df.loc[('Asia', 'China')] ---
Population       1412.0
Area_KM2      9596961.0
Name: (Asia, China), dtype: float64


#### Slicing MultiIndexed Rows
You can slice a range of rows using tuples:

In [11]:
# Slice from Asia, China up to Europe, Germany
sliced_df = df.loc[('Asia', 'China'):('Europe', 'Germany')]
print("--- Sliced MultiIndex ---")
print(sliced_df)

--- Sliced MultiIndex ---
                   Population  Area_KM2
Continent Country                      
Asia      China        1412.0   9596961
          India        1393.0   3287263
Europe    Albania         2.8     28748
          Germany        83.2    357022


#### D) Selection with `.iloc`

**CRUCIAL RULE**: `.iloc` uses **integer row positions** and does not understand MultiIndex labels. It sees the DataFrame as a flat list of rows starting at `0`:

In [12]:
# Get the first physical row (Africa, Angola)
first_row = df.iloc[0]
print("--- df.iloc[0] ---")
print(first_row)

--- df.iloc[0] ---
Population         32.8
Area_KM2      1246700.0
Name: (Africa, Angola), dtype: float64


### Common Beginner Mistakes
1.  **Omitting Tuples on Selection**: Writing `df.loc['Asia', 'China']` instead of `df.loc[('Asia', 'China')]`.
    *   *Why this fails:* Pandas interprets `df.loc[row, column]`. Therefore, `df.loc['Asia', 'China']` tells Pandas to look for a row named `'Asia'` and a column named `'China'`, which results in a `KeyError` since `'China'` is an index level, not a column name! **Always wrap multi-level index row keys in parentheses `()`**.

#### Exercise 1 (Hard)
Using the final sorted DataFrame from Module 32:
```text
                        Employees  Budget
Department  City
Engineering New York           45  450000
            London             38  400000
Sales       New York           12  150000
            London              8  110000
```
1. Select all records for the `'Sales'` department .
2. Select the row representing `'Engineering'` in `'London'`.
3. Select the 3rd physical row using `.iloc`.


In [13]:
# 1. Select all Sales department records
sales_dept = df_flat.loc['Sales']
print("--- Sales Department ---")
print(sales_dept)

# 2. Select Engineering in London
eng_london = df_flat.loc[('Engineering', 'London')]
print("--- Engineering London ---")
print(eng_london)

# 3. Select the 3rd physical row (index 2 is the 3rd row, Sales New York)
third_row = df_flat.iloc[2]
print("--- Third Row (.iloc[2]) ---")
print(third_row)

--- Sales Department ---
          index  Employees  Budget
City                              
New York      0         12  150000
London        2          8  110000
--- Engineering London ---
index             3
Employees        38
Budget       400000
Name: (Engineering, London), dtype: int64
--- Third Row (.iloc[2]) ---
index             0
Employees        12
Budget       150000
Name: (Sales, New York), dtype: int64
